# LegalBench Chain-of-Thought Language Experiment
## Does Reasoning Language Affect LLM Legal Accuracy?

This notebook analyzes results from the multilingual CoT experiment:
- **21 reasoning conditions** (16 natural languages + 3 abstract formats + wildcard + no-CoT)
- **6 LLMs** (Claude, GPT-4o, Mistral, DeepSeek, Qwen, Gemini)
- **10 LegalBench tasks** across evidence, contract, statutory, and consumer law

---
**Prerequisites:** Run `python run_experiment.py` first to generate results in `results/experiment_summary.json`.

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats as scipy_stats

# Style setup
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 200,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.family': 'sans-serif',
    'axes.titleweight': 'bold',
    'axes.titlesize': 13,
})

# Project imports
sys.path.insert(0, '.')
from config import MODELS, CONDITIONS, LEGALBENCH_TASKS

print(f'Models: {len(MODELS)} | Conditions: {len(CONDITIONS)} | Tasks: {len(LEGALBENCH_TASKS)}')
print(f'Full matrix: {len(MODELS)} x {len(CONDITIONS)} x {len(LEGALBENCH_TASKS)} = {len(MODELS)*len(CONDITIONS)*len(LEGALBENCH_TASKS)} cells')

## 1. Load Experiment Data

Loads results from `results/experiment_summary.json` produced by `run_experiment.py`.

In [ ]:
def load_data():
    """Load experiment results from results/experiment_summary.json."""
    results_path = Path('results/experiment_summary.json')
    
    if not results_path.exists():
        raise FileNotFoundError(
            "No experiment results found at results/experiment_summary.json\n\n"
            "Run the experiment first:\n"
            "  python run_experiment.py              # full run\n"
            "  python run_experiment.py --pilot       # quick 7-condition pilot\n"
            "  python run_experiment.py --runs 1      # single run (no variance)\n"
        )
    
    with open(results_path) as f:
        data = json.load(f)
    
    df = pd.DataFrame(data['results'])
    print(f'Loaded {len(df)} result cells')
    print(f'Models: {df["model"].nunique()} | Conditions: {df["condition"].nunique()} | Tasks: {df["task"].nunique()}')
    return df


df = load_data()
df.head(3)

## 2. Color Palettes & Helpers

In [ ]:
# Consistent color mapping
MODEL_COLORS = {
    'claude-sonnet': '#D97706',
    'gpt-4o': '#10B981',
    'gemini-2.5-flash': '#4285F4',
    'mistral-large': '#F97316',
    'deepseek-v3': '#6366F1',
    'qwen-max': '#EC4899',
}

FAMILY_COLORS = {
    'Indo-European': '#3B82F6',
    'Sino-Tibetan': '#EF4444',
    'Afroasiatic': '#F59E0B',
    'Japonic': '#EC4899',
    'Koreanic': '#8B5CF6',
    'Turkic': '#14B8A6',
    'Uralic': '#6366F1',
    'Austronesian': '#84CC16',
    'Austroasiatic': '#F97316',
    'Abstract': '#6B7280',
    'Wildcard': '#FBBF24',
    'Control': '#9CA3AF',
}

CONDITION_ORDER = [
    'english', 'wildcard', 'pseudocode', 'formal_logic', 'german', 'mandarin',
    'japanese', 'indonesian', 'russian', 'korean', 'vietnamese', 'turkish',
    'arabic', 'hebrew', 'emergent', 'hindi', 'finnish', 'hungarian', 'no_cot',
]

def format_pct(x, _=None):
    return f'{x:.0%}'

def save_fig(fig, name):
    Path('figures').mkdir(exist_ok=True)
    fig.savefig(f'figures/{name}.png', bbox_inches='tight', facecolor='white')

print('Palettes ready.')

---
## 3. Condition Ranking — Which Reasoning Language Wins?

In [ ]:
# Aggregate: mean accuracy per condition (across all models, tasks, runs)
cond_agg = (
    df.groupby(['condition', 'condition_family'])['accuracy']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .sort_values('mean', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 8))

colors = [FAMILY_COLORS.get(row.condition_family, '#666') for _, row in cond_agg.iterrows()]
bars = ax.barh(cond_agg['condition'], cond_agg['mean'], xerr=cond_agg['std'],
               color=colors, edgecolor='white', linewidth=0.5, capsize=3, error_kw={'linewidth': 1})

# Highlight top 3
top3 = cond_agg.nlargest(3, 'mean')['condition'].values
for bar, cond in zip(bars, cond_agg['condition']):
    if cond in top3:
        bar.set_edgecolor('#000')
        bar.set_linewidth(1.5)

ax.set_xlabel('Mean Accuracy')
ax.set_title('Reasoning Language Ranking — All Models Averaged')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlim(left=max(0, cond_agg['mean'].min() - 0.08))

# Add value labels
for bar, val in zip(bars, cond_agg['mean']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=8, fontweight='bold')

# Legend for families
from matplotlib.patches import Patch
families_present = cond_agg['condition_family'].unique()
legend_patches = [Patch(facecolor=FAMILY_COLORS.get(f, '#666'), label=f) for f in sorted(families_present)]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8, title='Language Family', title_fontsize=9)

plt.tight_layout()
save_fig(fig, 'condition_ranking')
plt.show()

---
## 4. Full Heatmap — Model x Condition Accuracy Matrix

In [ ]:
# Pivot: model x condition, averaged over tasks and runs
pivot = df.groupby(['model', 'condition'])['accuracy'].mean().unstack()

# Reorder conditions by overall mean
cond_order = pivot.mean().sort_values(ascending=False).index
model_order = pivot.mean(axis=1).sort_values(ascending=False).index
pivot = pivot.loc[model_order, cond_order]

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    pivot, annot=True, fmt='.1%', cmap='RdYlGn', center=pivot.values.mean(),
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'label': 'Accuracy', 'format': mticker.PercentFormatter(1.0)},
    annot_kws={'fontsize': 8},
)

# Pretty labels
display_names = {k: v['display'].split('(')[0].strip() for k, v in MODELS.items()}
ax.set_yticklabels([display_names.get(m, m) for m in model_order], rotation=0)
ax.set_xticklabels([c.replace('_', ' ').title() for c in cond_order], rotation=45, ha='right')
ax.set_title('Accuracy Heatmap — Model x Reasoning Condition')
ax.set_xlabel('')
ax.set_ylabel('')

plt.tight_layout()
save_fig(fig, 'heatmap_model_condition')
plt.show()

---
## 5. Model Comparison — Performance Across Conditions

In [ ]:
model_agg = (
    df.groupby('model')['accuracy']
    .agg(['mean', 'std'])
    .sort_values('mean', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 2]})

# Left: bar chart of overall model accuracy
ax = axes[0]
colors = [MODEL_COLORS.get(m, '#666') for m in model_agg['model']]
ax.barh(model_agg['model'], model_agg['mean'], xerr=model_agg['std'],
        color=colors, edgecolor='white', capsize=3)
ax.set_xlabel('Mean Accuracy')
ax.set_title('Overall Model Ranking')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlim(left=max(0, model_agg['mean'].min() - 0.06))
for i, (_, row) in enumerate(model_agg.iterrows()):
    ax.text(row['mean'] + 0.005, i, f'{row["mean"]:.1%}', va='center', fontsize=9, fontweight='bold')

# Right: violin plot showing distribution per model
ax = axes[1]
model_order_list = model_agg['model'].tolist()
palette = {m: MODEL_COLORS.get(m, '#666') for m in model_order_list}
sns.violinplot(data=df, y='model', x='accuracy', order=model_order_list,
               palette=palette, inner='quart', ax=ax, cut=0, linewidth=0.8)
ax.set_xlabel('Accuracy Distribution')
ax.set_ylabel('')
ax.set_title('Accuracy Distribution per Model')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.tight_layout()
save_fig(fig, 'model_comparison')
plt.show()

---
## 6. Language Family Analysis

In [ ]:
family_agg = (
    df.groupby('condition_family')['accuracy']
    .agg(['mean', 'std', 'count'])
    .sort_values('mean', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: family bar chart
ax = axes[0]
colors = [FAMILY_COLORS.get(f, '#666') for f in family_agg['condition_family']]
bars = ax.bar(range(len(family_agg)), family_agg['mean'], yerr=family_agg['std'],
              color=colors, edgecolor='white', capsize=3, linewidth=0.5)
ax.set_xticks(range(len(family_agg)))
ax.set_xticklabels(family_agg['condition_family'], rotation=40, ha='right')
ax.set_ylabel('Mean Accuracy')
ax.set_title('Accuracy by Language Family')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
for bar, val in zip(bars, family_agg['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.008,
            f'{val:.1%}', ha='center', fontsize=8, fontweight='bold')

# Right: boxplot per family per model
ax = axes[1]
# Filter to natural language families only (skip Control, Wildcard, Abstract for clarity)
nat_families = ['Indo-European', 'Sino-Tibetan', 'Afroasiatic', 'Japonic',
                'Koreanic', 'Turkic', 'Uralic', 'Austronesian', 'Austroasiatic']
df_nat = df[df['condition_family'].isin(nat_families)]
family_order = (
    df_nat.groupby('condition_family')['accuracy'].mean()
    .sort_values(ascending=False).index.tolist()
)
nat_palette = {f: FAMILY_COLORS.get(f, '#666') for f in family_order}
sns.boxplot(data=df_nat, x='condition_family', y='accuracy', order=family_order,
            palette=nat_palette, ax=ax, linewidth=0.8, fliersize=2)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
ax.set_ylabel('Accuracy')
ax.set_xlabel('')
ax.set_title('Natural Language Families — Distribution')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.tight_layout()
save_fig(fig, 'language_family')
plt.show()

---
## 7. English vs. Wildcard vs. No-CoT — The Key Comparison

In [ ]:
key_conditions = ['english', 'wildcard', 'pseudocode', 'formal_logic', 'no_cot']
df_key = df[df['condition'].isin(key_conditions)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: grouped bar by model
ax = axes[0]
key_pivot = df_key.groupby(['model', 'condition'])['accuracy'].mean().unstack()
key_pivot = key_pivot[key_conditions]
key_pivot = key_pivot.loc[key_pivot.mean(axis=1).sort_values(ascending=False).index]

cond_colors = {
    'english': '#3B82F6', 'wildcard': '#FBBF24', 'pseudocode': '#6B7280',
    'formal_logic': '#9CA3AF', 'no_cot': '#EF4444'
}
key_pivot.plot.bar(ax=ax, color=[cond_colors[c] for c in key_conditions],
                   edgecolor='white', linewidth=0.5, width=0.75)
ax.set_ylabel('Mean Accuracy')
ax.set_title('Key Conditions — Per Model')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(title='Condition', fontsize=8, title_fontsize=9, loc='lower right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_xlabel('')

# Right: delta from English baseline per model
ax = axes[1]
english_acc = df[df['condition'] == 'english'].groupby('model')['accuracy'].mean()
deltas = []
for cond in ['wildcard', 'pseudocode', 'formal_logic', 'no_cot']:
    cond_acc = df[df['condition'] == cond].groupby('model')['accuracy'].mean()
    for model in cond_acc.index:
        deltas.append({
            'model': model,
            'condition': cond,
            'delta': cond_acc[model] - english_acc.get(model, 0)
        })
df_delta = pd.DataFrame(deltas)

delta_pivot = df_delta.pivot(index='model', columns='condition', values='delta')
delta_pivot = delta_pivot[['wildcard', 'pseudocode', 'formal_logic', 'no_cot']]
delta_pivot = delta_pivot.loc[delta_pivot.mean(axis=1).sort_values(ascending=False).index]

delta_pivot.plot.barh(ax=ax, color=[cond_colors[c] for c in delta_pivot.columns],
                      edgecolor='white', linewidth=0.5)
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='-')
ax.set_xlabel('Accuracy Delta vs. English Baseline')
ax.set_title('Performance Delta from English CoT')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(title='Condition', fontsize=8, title_fontsize=9)
ax.set_ylabel('')

plt.tight_layout()
save_fig(fig, 'english_vs_wildcard')
plt.show()

---
## 8. Token Efficiency — Accuracy per 1K Reasoning Tokens

In [ ]:
token_agg = (
    df.groupby('condition')
    .agg(avg_accuracy=('accuracy', 'mean'), avg_tokens=('avg_output_tokens', 'mean'))
    .reset_index()
)
token_agg['efficiency'] = token_agg['avg_accuracy'] / token_agg['avg_tokens'] * 1000
token_agg['family'] = token_agg['condition'].map(lambda c: CONDITIONS.get(c, {}).get('family', '?'))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: scatter — accuracy vs tokens
ax = axes[0]
colors = [FAMILY_COLORS.get(f, '#666') for f in token_agg['family']]
scatter = ax.scatter(token_agg['avg_tokens'], token_agg['avg_accuracy'],
                     c=colors, s=100, edgecolors='white', linewidth=0.8, zorder=3)
for _, row in token_agg.iterrows():
    ax.annotate(row['condition'], (row['avg_tokens'], row['avg_accuracy']),
                fontsize=7, ha='center', va='bottom', xytext=(0, 5),
                textcoords='offset points')
ax.set_xlabel('Avg Output Tokens (Reasoning Length)')
ax.set_ylabel('Mean Accuracy')
ax.set_title('Accuracy vs. Reasoning Length')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Right: efficiency bar
ax = axes[1]
eff_sorted = token_agg.sort_values('efficiency', ascending=True)
colors = [FAMILY_COLORS.get(f, '#666') for f in eff_sorted['family']]
ax.barh(eff_sorted['condition'], eff_sorted['efficiency'], color=colors,
        edgecolor='white', linewidth=0.5)
ax.set_xlabel('Efficiency (Accuracy per 1K Tokens)')
ax.set_title('Token Efficiency Ranking')
for i, (_, row) in enumerate(eff_sorted.iterrows()):
    ax.text(row['efficiency'] + 0.02, i, f'{row["efficiency"]:.2f}',
            va='center', fontsize=8)

plt.tight_layout()
save_fig(fig, 'token_efficiency')
plt.show()

---
## 9. Training Data Origin Advantage

Do Chinese-origin models (DeepSeek, Qwen) get a boost when reasoning in Mandarin?  
Does Mistral (France) benefit from German (closest available)?

In [ ]:
# Test: Chinese models in Mandarin vs others in Mandarin
origin_tests = [
    ('Mandarin CoT — Chinese vs. Non-Chinese Models',
     'mandarin',
     {'deepseek-v3': 'DeepSeek (China)', 'qwen-max': 'Qwen (China)'},
     'China'),
    ('German CoT — Mistral vs. Others',
     'german',
     {'mistral-large': 'Mistral (France)'},
     'France'),
]

fig, axes = plt.subplots(1, len(origin_tests), figsize=(14, 5))
if len(origin_tests) == 1:
    axes = [axes]

for ax, (title, cond, advantage_models, origin) in zip(axes, origin_tests):
    df_cond = df[df['condition'] == cond]
    model_acc = df_cond.groupby('model')['accuracy'].mean().sort_values(ascending=False)
    
    colors = ['#EF4444' if m in advantage_models else '#94A3B8' for m in model_acc.index]
    bars = ax.bar(range(len(model_acc)), model_acc.values, color=colors,
                  edgecolor='white', linewidth=0.5)
    ax.set_xticks(range(len(model_acc)))
    labels = []
    for m in model_acc.index:
        if m in advantage_models:
            labels.append(f'{advantage_models[m]}')
        else:
            labels.append(m)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Mean Accuracy')
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    
    for bar, val in zip(bars, model_acc.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
                f'{val:.1%}', ha='center', fontsize=9, fontweight='bold')
    
    # Add annotation for advantage
    adv_accs = [model_acc[m] for m in advantage_models if m in model_acc]
    other_accs = [model_acc[m] for m in model_acc.index if m not in advantage_models]
    if adv_accs and other_accs:
        delta = np.mean(adv_accs) - np.mean(other_accs)
        verdict = 'CONFIRMED' if delta > 0.02 else ('MARGINAL' if delta > 0 else 'NOT FOUND')
        color = '#16A34A' if delta > 0.02 else ('#F59E0B' if delta > 0 else '#EF4444')
        ax.text(0.95, 0.05, f'Origin advantage: {verdict}\n({origin} models {delta:+.1%})',
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=9, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9))

plt.tight_layout()
save_fig(fig, 'origin_advantage')
plt.show()

---
## 10. Per-Task Breakdown — Which Tasks Benefit Most from Language Choice?

In [ ]:
# Task sensitivity: range of accuracy across conditions
task_cond = df.groupby(['task', 'condition'])['accuracy'].mean().unstack()
task_sensitivity = pd.DataFrame({
    'best_condition': task_cond.idxmax(axis=1),
    'best_acc': task_cond.max(axis=1),
    'worst_condition': task_cond.idxmin(axis=1),
    'worst_acc': task_cond.min(axis=1),
    'range': task_cond.max(axis=1) - task_cond.min(axis=1),
    'mean_acc': task_cond.mean(axis=1),
}).sort_values('range', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: task accuracy with range bars
ax = axes[0]
y_pos = range(len(task_sensitivity))
ax.barh(y_pos, task_sensitivity['range'], left=task_sensitivity['worst_acc'],
        color='#BFDBFE', edgecolor='#3B82F6', linewidth=0.8, height=0.6, label='Range')
ax.scatter(task_sensitivity['mean_acc'], y_pos, color='#1E40AF', zorder=5, s=40, label='Mean')
ax.set_yticks(y_pos)
short_names = [t.replace('contract_nli_', 'cnli_').replace('supply_chain_disclosure_best_practice_verification', 'supply_chain')
               .replace('learned_hands_', 'lh_') for t in task_sensitivity.index]
ax.set_yticklabels(short_names, fontsize=9)
ax.set_xlabel('Accuracy')
ax.set_title('Task Sensitivity to Reasoning Language')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=8)

# Right: heatmap of task x condition (top 8 conditions)
ax = axes[1]
top_conds = cond_agg.nlargest(10, 'mean')['condition'].tolist()
task_heat = task_cond[top_conds]
# Shorter task names for display
task_heat.index = [t.replace('contract_nli_', 'cnli_')
                   .replace('supply_chain_disclosure_best_practice_verification', 'supply_chain')
                   .replace('learned_hands_', 'lh_') for t in task_heat.index]
sns.heatmap(task_heat, annot=True, fmt='.0%', cmap='RdYlGn', ax=ax,
            linewidths=0.5, linecolor='white', annot_kws={'fontsize': 8},
            cbar_kws={'format': mticker.PercentFormatter(1.0), 'label': 'Accuracy'})
ax.set_title('Task x Top-10 Conditions')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
ax.set_ylabel('')

plt.tight_layout()
save_fig(fig, 'task_breakdown')
plt.show()

---
## 11. Radar Chart — Model Profiles Across Condition Families

In [ ]:
# Radar: each model's accuracy by condition family
radar_data = df.groupby(['model', 'condition_family'])['accuracy'].mean().unstack()
families = radar_data.columns.tolist()
n_families = len(families)

angles = np.linspace(0, 2 * np.pi, n_families, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for model in radar_data.index:
    values = radar_data.loc[model].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=1.5, markersize=4,
            label=model, color=MODEL_COLORS.get(model, '#666'))
    ax.fill(angles, values, alpha=0.08, color=MODEL_COLORS.get(model, '#666'))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(families, fontsize=8)
ax.set_title('Model Profiles by Language Family', y=1.08, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

plt.tight_layout()
save_fig(fig, 'radar_model_profiles')
plt.show()

---
## 12. Run Variance — How Stable Are Results?

In [ ]:
# Variance across runs for each model x condition cell
run_var = (
    df.groupby(['model', 'condition', 'task'])['accuracy']
    .std()
    .reset_index()
    .rename(columns={'accuracy': 'run_std'})
)
run_var['run_std'] = run_var['run_std'].fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution of per-cell std
ax = axes[0]
ax.hist(run_var['run_std'], bins=40, color='#6366F1', edgecolor='white', alpha=0.85)
ax.axvline(run_var['run_std'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Mean std: {run_var["run_std"].mean():.3f}')
ax.set_xlabel('Standard Deviation Across Runs')
ax.set_ylabel('Count (cells)')
ax.set_title('Run-to-Run Variance Distribution')
ax.legend()

# Right: avg variance by condition
ax = axes[1]
cond_var = run_var.groupby('condition')['run_std'].mean().sort_values(ascending=True)
cond_families = {c: CONDITIONS.get(c, {}).get('family', '?') for c in cond_var.index}
colors = [FAMILY_COLORS.get(cond_families[c], '#666') for c in cond_var.index]
ax.barh(cond_var.index, cond_var.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Mean Std Dev Across Runs')
ax.set_title('Variance by Condition (Lower = More Stable)')

plt.tight_layout()
save_fig(fig, 'variance_analysis')
plt.show()

---
## 13. Statistical Significance — Is Wildcard Really Better Than English?

In [ ]:
# Paired comparison: English vs Wildcard accuracy per (model, task, run)
english_scores = df[df['condition'] == 'english'].set_index(['model', 'task', 'run'])['accuracy']
wildcard_scores = df[df['condition'] == 'wildcard'].set_index(['model', 'task', 'run'])['accuracy']

# Align on shared index
shared_idx = english_scores.index.intersection(wildcard_scores.index)
eng = english_scores.loc[shared_idx].values
wc = wildcard_scores.loc[shared_idx].values

# Paired t-test
t_stat, p_value = scipy_stats.ttest_rel(wc, eng)
effect_size = (wc.mean() - eng.mean()) / np.std(wc - eng)  # Cohen's d for paired

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: distribution of differences
ax = axes[0]
diffs = wc - eng
ax.hist(diffs, bins=40, color='#FBBF24', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=1, linestyle='-')
ax.axvline(diffs.mean(), color='#EF4444', linewidth=2, linestyle='--',
           label=f'Mean diff: {diffs.mean():+.3f}')
ax.set_xlabel('Wildcard - English Accuracy')
ax.set_ylabel('Count')
ax.set_title('Wildcard vs. English — Paired Differences')
ax.legend()

sig = 'YES' if p_value < 0.05 else 'NO'
color = '#16A34A' if p_value < 0.05 else '#EF4444'
ax.text(0.95, 0.95,
        f'Paired t-test\nt = {t_stat:.2f}\np = {p_value:.4f}\nCohen\'s d = {effect_size:.3f}\nSignificant: {sig}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=color, alpha=0.95))

# Right: scatter of paired values
ax = axes[1]
ax.scatter(eng, wc, alpha=0.3, s=15, color='#6366F1', edgecolors='none')
lims = [min(eng.min(), wc.min()) - 0.05, max(eng.max(), wc.max()) + 0.05]
ax.plot(lims, lims, 'k--', linewidth=0.8, alpha=0.5, label='y = x')
ax.set_xlabel('English Accuracy')
ax.set_ylabel('Wildcard Accuracy')
ax.set_title('Paired Comparison: Wildcard vs. English')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')
ax.legend(fontsize=9)

# Points above line = wildcard wins
above = (wc > eng).sum()
below = (wc < eng).sum()
tied = (wc == eng).sum()
ax.text(0.05, 0.95, f'Wildcard wins: {above}\nEnglish wins: {below}\nTied: {tied}',
        transform=ax.transAxes, ha='left', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#6366F1', alpha=0.9))

plt.tight_layout()
save_fig(fig, 'statistical_significance')
plt.show()

---
## 14. Executive Summary Table

In [ ]:
# Build a clean summary table
summary = (
    df.groupby('condition')
    .agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        avg_tokens=('avg_output_tokens', 'mean'),
        n_cells=('accuracy', 'count'),
    )
    .reset_index()
)
summary['family'] = summary['condition'].map(lambda c: CONDITIONS.get(c, {}).get('family', '?'))
summary['efficiency'] = summary['mean_accuracy'] / summary['avg_tokens'] * 1000
summary['delta_vs_english'] = summary['mean_accuracy'] - summary.loc[
    summary['condition'] == 'english', 'mean_accuracy'
].values[0]

summary = summary.sort_values('mean_accuracy', ascending=False)

# Display styled
styled = (
    summary[['condition', 'family', 'mean_accuracy', 'std_accuracy',
             'delta_vs_english', 'avg_tokens', 'efficiency']]
    .style
    .format({
        'mean_accuracy': '{:.1%}',
        'std_accuracy': '{:.3f}',
        'delta_vs_english': '{:+.1%}',
        'avg_tokens': '{:.0f}',
        'efficiency': '{:.2f}',
    })
    .bar(subset=['mean_accuracy'], color='#BFDBFE', vmin=0.4, vmax=1.0)
    .bar(subset=['delta_vs_english'], color=['#FCA5A5', '#BBF7D0'], align='zero')
    .set_caption('Condition Summary — Sorted by Accuracy')
)
styled

---
## 15. Key Findings

In [ ]:
from IPython.display import Markdown, display

best_cond = summary.iloc[0]
worst_cond = summary.iloc[-1]
eng = summary[summary['condition'] == 'english'].iloc[0]
wc_row = summary[summary['condition'] == 'wildcard'].iloc[0] if 'wildcard' in summary['condition'].values else None
nocot = summary[summary['condition'] == 'no_cot'].iloc[0] if 'no_cot' in summary['condition'].values else None

best_model = model_agg.iloc[0]

findings = f"""
### Key Findings

| Metric | Value |
|--------|-------|
| **Best reasoning condition** | {best_cond['condition']} ({best_cond['mean_accuracy']:.1%}) |
| **Worst reasoning condition** | {worst_cond['condition']} ({worst_cond['mean_accuracy']:.1%}) |
| **English baseline** | {eng['mean_accuracy']:.1%} |
| **Wildcard (unconstrained)** | {wc_row['mean_accuracy']:.1%} ({wc_row['delta_vs_english']:+.1%} vs English) | 
| **No-CoT (control)** | {nocot['mean_accuracy']:.1%} ({nocot['delta_vs_english']:+.1%} vs English) |
| **Best model overall** | {best_model['model']} ({best_model['mean']:.1%}) |
| **CoT value** | {eng['mean_accuracy'] - nocot['mean_accuracy']:.1%} improvement over no-CoT |
| **Accuracy spread** | {best_cond['mean_accuracy'] - worst_cond['mean_accuracy']:.1%} (best - worst condition) |

#### Hypotheses

{'- **Wildcard > English: SUPPORTED** — Unconstrained polyglot reasoning outperforms enforced English CoT' if wc_row is not None and wc_row['delta_vs_english'] > 0.005 else '- **Wildcard > English: NOT SUPPORTED** — English CoT is competitive with unconstrained reasoning'}
{'- **CoT matters: STRONGLY SUPPORTED** — Chain-of-thought provides substantial accuracy gains over direct answering' if nocot is not None and nocot['delta_vs_english'] < -0.05 else '- **CoT impact: MODEST**'}
- **Language family matters:** {summary.groupby('family')['mean_accuracy'].mean().idxmax()} family performs best overall
- **Most token-efficient:** {summary.sort_values('efficiency', ascending=False).iloc[0]['condition']} ({summary.sort_values('efficiency', ascending=False).iloc[0]['efficiency']:.2f} acc/1K tokens)
"""

display(Markdown(findings))

---
## 16. Export

In [ ]:
# Save CSV for further analysis
Path('results').mkdir(exist_ok=True)
df.to_csv('results/results_matrix.csv', index=False)
summary.to_csv('results/condition_summary.csv', index=False)

print('Exported:')
print('  results/results_matrix.csv — full experiment data')
print('  results/condition_summary.csv — condition-level summary')
print('  figures/*.png — all charts')